# Contract Intelligence Multi-Agent System
## Assignment Notebook - 8-Week Capstone Project

**Course**: Agentic AI Bootcamp - Staff-Level System Design

---

## Welcome to Your Capstone Assignment!

This notebook is your guided journey to building a **production-grade multi-agent contract intelligence system**. Unlike the master solution, YOU will implement the core functionality.

### How This Assignment Works

1. **Scaffolding Provided**: Class structures, function signatures, and imports are given
2. **TODO Markers**: Look for `# TODO:` comments - these are YOUR tasks
3. **Hints**: Each section has hints to guide you (but not give away the answer)
4. **Validation Cells**: Run these to check if your implementation is correct
5. **Expected Output**: Sample outputs show what success looks like

### Difficulty Progression

| Week | Difficulty | Focus |
|------|-----------|-------|
| 1 | Easy | Environment setup (mostly provided) |
| 2 | Easy-Medium | Document processing methods |
| 3 | Medium | Vector store search functions |
| 4 | Medium-Hard | Implement agents from scratch |
| 5 | Hard | Multi-agent orchestration |
| 6 | Medium | Build knowledge graph |
| 7 | Medium | Add observability metrics |
| 8 | Hard | Integrate everything |

### Grading Rubric

- **Week 1-2**: 15% (Foundation)
- **Week 3-4**: 30% (Core Components)  
- **Week 5-6**: 30% (Advanced Features)
- **Week 7-8**: 25% (Production Polish)

Let's begin!

---

# WEEK 1: Environment & Foundations

**Difficulty: Easy** - Most code is provided. Focus on understanding.

---

## Learning Objectives

By the end of Week 1, you will:
- [ ] Set up all required packages
- [ ] Configure API keys securely
- [ ] Initialize Langfuse for observability
- [ ] Create traced wrapper functions
- [ ] Explore the contract data structure

---

## 1.1 Package Installation

This section is **provided** - just run the cell to install packages.

In [1]:
# ============================================================================
# WEEK 1.1: PACKAGE INSTALLATION (PROVIDED)
# ============================================================================
# Pinned versions for reproducibility. Every package is verified after install.

import subprocess, sys

_packages = ["openai==1.59.6", "langfuse==2.57.1", "langchain==0.3.14", "langchain-openai==0.2.14", "langchain-community==0.3.14", "langchain-core==0.3.29", "chromadb", "networkx", "pyvis", "python-docx", "openpyxl", "plotly", "seaborn", "pydantic>=2.0", "tenacity", "rich", "gradio", "python-dotenv", "numpy", "pandas", "matplotlib"]

print("Installing packages (this may take 1-2 minutes)...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + _packages,
    capture_output=True, text=True
)

if result.returncode != 0:
    for line in result.stderr.split("\n"):
        if line.strip() and "dependency resolver" not in line.lower() and "notice" not in line.lower():
            print(line)

# ---------- Verify EVERY import ----------
_verify = [
    ("openai", "openai"),
    ("langfuse", "langfuse"),
    ("langchain", "langchain"),
    ("langchain_openai", "langchain_openai"),
    ("chromadb", "chromadb"),
    ("networkx", "networkx"),
    ("pyvis", "pyvis.network"),
    ("docx", "docx"),
    ("openpyxl", "openpyxl"),
    ("plotly", "plotly"),
    ("seaborn", "seaborn"),
    ("pydantic", "pydantic"),
    ("tenacity", "tenacity"),
    ("rich", "rich"),
    ("gradio", "gradio"),
    ("dotenv", "dotenv"),
    ("fastapi", "fastapi"),
    ("uvicorn", "uvicorn"),
]

_failed = []
for name, imp in _verify:
    try:
        __import__(imp)
    except ImportError:
        _failed.append(name)

if _failed:
    msg = f"FATAL: These packages failed to import: {', '.join(_failed)}\n"
    msg += "Try: Runtime > Restart runtime, then re-run this cell."
    raise ImportError(msg)

print("=" * 60)
print(f"All {len(_verify)} packages installed and verified!")
print("=" * 60)

Installing packages (this may take 1-2 minutes)...


e:\Sheriff_faang\full_end_to_end_project_implementation\IK_PWC_Agentic_AI_Project\PWC_CAPSTONE_PROJECT\IK_PWC_COURSE_CAPSTONE_PROJECTS\.contract\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All 18 packages installed and verified!


## 1.2 Environment Configuration

This section is **provided** - handles environment detection and imports.

In [2]:
# ============================================================================
# WEEK 1.2: ENVIRONMENT DETECTION (PROVIDED)
# ============================================================================

import os
import sys
import json
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Optional, Any, Tuple
from dataclasses import dataclass, field
from enum import Enum
import warnings
warnings.filterwarnings('ignore')

# Environment detection
IN_COLAB = 'google.colab' in sys.modules

print(f"Runtime Environment: {'Google Colab' if IN_COLAB else 'Local/Other'}")
print(f"Python Version: {sys.version.split()[0]}")

Runtime Environment: Local/Other
Python Version: 3.12.7


In [4]:
# ============================================================================
# API KEY CONFIGURATION (PROVIDED)
# ============================================================================

import subprocess

DATASET_REPO = "https://github.com/AI-Project-Lab/IK-pwc-agenticai-datasets.git"
DATASET_PROJECT = "contract_intelligence"

if IN_COLAB:
    print("Configuring for Google Colab environment...")

    # --- API Key Configuration ---
    try:
        from google.colab import userdata
        os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
        os.environ['LANGFUSE_SECRET_KEY'] = userdata.get('LANGFUSE_SECRET_KEY')
        os.environ['LANGFUSE_PUBLIC_KEY'] = userdata.get('LANGFUSE_PUBLIC_KEY')
        _host = userdata.get('LANGFUSE_HOST') or ''
        os.environ['LANGFUSE_HOST'] = _host if _host.startswith('http') else 'https://cloud.langfuse.com'
        print("API keys loaded from Colab Secrets")
    except Exception as e:
        print(f"Colab Secrets not available: {e}")
        import getpass
        os.environ['OPENAI_API_KEY'] = getpass.getpass('Enter OpenAI API Key: ')
        os.environ['LANGFUSE_SECRET_KEY'] = getpass.getpass('Enter Langfuse Secret Key: ')
        os.environ['LANGFUSE_PUBLIC_KEY'] = getpass.getpass('Enter Langfuse Public Key: ')
        os.environ['LANGFUSE_HOST'] = 'https://cloud.langfuse.com'

    # --- Dataset Ingestion from GitHub ---
    dataset_path = "/content/datasets"
    if not os.path.exists(f"{dataset_path}/{DATASET_PROJECT}"):
        print(f"\nCloning dataset from {DATASET_REPO}...")
        subprocess.run(["git", "clone", "--depth", "1", DATASET_REPO, dataset_path],
                       check=True, capture_output=True)
        print("Dataset cloned successfully!")
    else:
        print("\nDataset already available.")
    DATA_DIR_BASE = f"{dataset_path}/{DATASET_PROJECT}"
else:
    print("Configuring for local environment...")
    from dotenv import load_dotenv
    load_dotenv()

    # --- Dataset Ingestion from GitHub ---
    datasets_parent = Path('.').resolve().parent.parent / 'datasets'
    if not (datasets_parent / DATASET_PROJECT).exists():
        print(f"\nCloning dataset from {DATASET_REPO}...")
        subprocess.run(["git", "clone", "--depth", "1", DATASET_REPO, str(datasets_parent)],
                       check=True, capture_output=True)
        print("Dataset cloned successfully!")
    else:
        print("\nDataset already available locally.")
    DATA_DIR_BASE = str(datasets_parent / DATASET_PROJECT)

# Validate API keys
required_keys = ['OPENAI_API_KEY', 'LANGFUSE_SECRET_KEY', 'LANGFUSE_PUBLIC_KEY']
missing_keys = [key for key in required_keys if not os.environ.get(key)]
if missing_keys:
    raise EnvironmentError(f"Missing required API keys: {missing_keys}")

PROJECT_NAME = "contract-intelligence-system"
DATA_DIR = Path(DATA_DIR_BASE)
print(f"\nAll API keys validated successfully")
print(f"Data directory: {DATA_DIR}")
print(f"Directory exists: {DATA_DIR.exists()}")
if DATA_DIR.exists():
    file_count = sum(1 for _ in DATA_DIR.rglob('*') if _.is_file())
    print(f"Total files found: {file_count}")

Configuring for local environment...

Dataset already available locally.

All API keys validated successfully
Data directory: E:\Sheriff_faang\full_end_to_end_project_implementation\IK_PWC_Agentic_AI_Project\PWC_CAPSTONE_PROJECT\IK_PWC_COURSE_CAPSTONE_PROJECTS\datasets\contract_intelligence
Directory exists: True
Total files found: 36


## 1.3 Langfuse Initialization

**YOUR FIRST TODO!** Initialize the Langfuse client.

### Hints:
- Import `Langfuse` from `langfuse`
- Import `observe` and `langfuse_context` from `langfuse.decorators`
- Use environment variables for credentials
- Call `auth_check()` to verify connection

In [5]:
# ============================================================================
# WEEK 1.3: LANGFUSE INITIALIZATION
# ============================================================================
# TODO: Import necessary Langfuse modules

from dotenv import load_dotenv
load_dotenv()

from langfuse import Langfuse
from langfuse.decorators import observe, langfuse_context
from openai import OpenAI

# TODO: Initialize the Langfuse client
# Hint: Use os.environ.get() to retrieve keys
# The constructor takes: secret_key, public_key, host
langfuse = Langfuse(
    secret_key=os.environ.get("LANGFUSE_SECRET_KEY"),
    public_key=os.environ.get("LANGFUSE_PUBLIC_KEY"),
    host=os.environ.get("LANGFUSE_HOST", "https://cloud.langfuse.com")
)


# TODO: Verify connection
# Hint: Use langfuse.auth_check() in a try/except block

# YOUR CODE HERE:
try:
    langfuse.auth_check()
    print("Langfuse connection verified!")
except Exception as error:
    raise ConnectionError(
        "Langfuse connection failed. Check LANGFUSE_SECRET_KEY, "
        "LANGFUSE_PUBLIC_KEY, and LANGFUSE_HOST in your .env file. "
        f"Details: {error}"
    ) from error


# TODO: Create a unique session ID
# Hint: Use datetime.now().strftime() to create a unique identifier
# Format: "contract-intel-YYYYMMDD-HHMMSS"

SESSION_ID = f"contract-intel-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

# YOUR CODE HERE:


print(f"Session ID: {SESSION_ID}")

Langfuse connection verified!
Session ID: contract-intel-20260912-005050


### Expected Output for 1.3:
```
Langfuse connection verified!
Session ID: contract-intel-20250127-143052
```

## 1.4 Traced Wrapper Functions

**TODO:** Implement traced versions of OpenAI calls.

### Hints:
- Create a trace with `langfuse.trace()`
- Create a generation span with `trace.generation()`
- Use `openai_client.embeddings.create()` for embeddings
- End the generation with usage stats

In [6]:
# ============================================================================
# WEEK 1.4: TRACED EMBEDDING FUNCTION
# ============================================================================

openai_client = OpenAI()

def traced_embedding(text: str, trace_name: str = "embedding") -> List[float]:
    """Generate an OpenAI embedding with Langfuse tracing."""
    if not text or not text.strip():
        raise ValueError("Cannot create an embedding for empty text.")

    trace = langfuse.trace(
        name=trace_name,
        session_id=SESSION_ID,
        metadata={
            "text_length": len(text),
            "text_preview": text[:100] + "..." if len(text) > 100 else text
        }
    )

    generation = trace.generation(
        name="openai-embedding",
        model="text-embedding-3-small",
        input=text[:500] + "..." if len(text) > 500 else text
    )

    try:
        response = openai_client.embeddings.create(
            model="text-embedding-3-small",
            input=text
        )
        embedding = response.data[0].embedding

        generation.end(
            output={"dimensions": len(embedding)},
            usage={"total_tokens": getattr(response.usage, "total_tokens", 0)}
        )

        return embedding
    except Exception as error:
        generation.end(output={"error": str(error)})
        trace.update(metadata={"status": "failed"})
        raise

print("traced_embedding() function defined")

traced_embedding() function defined


In [7]:
# ============================================================================
# WEEK 1.4: TRACED COMPLETION FUNCTION
# ============================================================================

def traced_completion(prompt: str, trace_name: str = "completion") -> str:
    """
    Send a prompt to OpenAI and trace the full LLM call in Langfuse.

    Args:
        prompt: User instruction or question sent to the language model.
        trace_name: Name shown for this operation in Langfuse.

    Returns:
        The text response generated by gpt-4o-mini.
    """

    # Validate input before sending an API request.
    if not prompt or not prompt.strip():
        raise ValueError("Cannot generate a completion for an empty prompt.")

    # Trace represents the full completion workflow.
    # Metadata stores only prompt length and a short preview.
    trace = langfuse.trace(
        name=trace_name,
        session_id=SESSION_ID,
        metadata={
            "prompt_length": len(prompt),
            "prompt_preview": prompt[:100] + "..." if len(prompt) > 100 else prompt
        }
    )

    # Generation tracks the actual model invocation.
    generation = trace.generation(
        name="openai-chat-completion",
        model="gpt-4o-mini",
        input=prompt[:500] + "..." if len(prompt) > 500 else prompt
    )

    try:
        # Send the prompt to OpenAI.
        response = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0
        )

        # Read generated text safely.
        response_text = response.choices[0].message.content or ""

        # Record model output and token usage in Langfuse.
        generation.end(
            output=response_text,
            usage={
                "prompt_tokens": getattr(response.usage, "prompt_tokens", 0),
                "completion_tokens": getattr(response.usage, "completion_tokens", 0),
                "total_tokens": getattr(response.usage, "total_tokens", 0)
            }
        )

        return response_text

    except Exception as error:
        # Keep the error traceable, then let the notebook show the real error.
        generation.end(output={"error": str(error)})
        trace.update(metadata={"status": "failed"})
        raise


print("traced_completion() function defined")

traced_completion() function defined


## 1.5 Contract Document Taxonomy

**PROVIDED** - Study this structure carefully, you'll need it later.

In [12]:
# ============================================================================
# WEEK 1.5: CONTRACT TAXONOMY (PROVIDED)
# ============================================================================

CONTRACT_CATEGORIES = {
    'master_agreements': {
        'path': 'Master level agreements',
        'description': 'Foundation contracts establishing overall relationship',
        'document_types': ['MSA', 'NDA', 'Rate Cards'],
        'risk_focus': ['legal', 'compliance'],
    },
    'transaction_contracts': {
        'path': 'Transaction level contract',
        'description': 'Project-specific agreements under master agreements',
        'document_types': ['SOW', 'Work Orders', 'Renewals', 'Amendments'],
        'risk_focus': ['operational', 'financial'],
    },
    'commercial_docs': {
        'path': 'Commercial docs',
        'description': 'Pricing and commercial terms documentation',
        'document_types': ['Pricing Tables', 'Rate Schedules'],
        'risk_focus': ['financial'],
    },
    'financial_billing': {
        'path': 'Financial and Billing docs',
        'description': 'Invoices, payment records, and billing schedules',
        'document_types': ['Invoices', 'Credit Notes', 'Payment Schedules'],
        'risk_focus': ['financial'],
    },
    'operational_docs': {
        'path': 'Operational service delivery docs',
        'description': 'Service delivery and operational documentation',
        'document_types': ['Service Reports', 'SLA Reports'],
        'risk_focus': ['operational'],
    },
    'compliance_docs': {
        'path': 'Compliance and policy documents',
        'description': 'Security, compliance, and policy documentation',
        'document_types': ['InfoSec Controls', 'Vendor Policies'],
        'risk_focus': ['compliance'],
    },
    'historical_negotiation': {
        'path': 'Historical negotiation data',
        'description': 'Historical negotiation records and commercial discussions',
        'document_types': ['Negotiation History', 'Commercial Correspondence'],
        'risk_focus': ['financial', 'legal'],
    },
    'disputes_audits_governance': {
        'path': 'Disputes, audits and governance data',
        'description': 'Dispute, audit, and governance records',
        'document_types': ['Dispute Records', 'Audit Reports', 'Governance Documents'],
        'risk_focus': ['legal', 'compliance', 'operational'],
    }
}

print(f"Contract categories defined: {len(CONTRACT_CATEGORIES)}")
for cat, info in CONTRACT_CATEGORIES.items():
    print(f"  - {cat}: {info['description'][:50]}...")

Contract categories defined: 8
  - master_agreements: Foundation contracts establishing overall relation...
  - transaction_contracts: Project-specific agreements under master agreement...
  - commercial_docs: Pricing and commercial terms documentation...
  - financial_billing: Invoices, payment records, and billing schedules...
  - operational_docs: Service delivery and operational documentation...
  - compliance_docs: Security, compliance, and policy documentation...
  - historical_negotiation: Historical negotiation records and commercial disc...
  - disputes_audits_governance: Dispute, audit, and governance records...


## 1.6 Data Discovery

**TODO:** Implement the data discovery function.

### Hints:
- Use `Path.rglob('*')` to recursively find files
- Check file suffixes with `item.suffix.lower()`
- Create spans for each category scan

In [13]:
# ============================================================================
# WEEK 1.6: DATA DISCOVERY FUNCTION
# ============================================================================

def discover_contract_data(data_dir: Path) -> Dict[str, List[Path]]:
    """Discover contract documents by taxonomy category and trace the scan."""
    trace = langfuse.trace(
        name="data-discovery",
        session_id=SESSION_ID,
        input={"data_dir": str(data_dir)}
    )

    discovered = {category: [] for category in CONTRACT_CATEGORIES}

    if not data_dir.exists():
        trace.update(
            output={
                "status": "failed",
                "reason": f"Data directory not found: {data_dir}"
            }
        )
        print(f"Warning: data directory not found: {data_dir}")
        return discovered

    supported_extensions = {".docx", ".pdf", ".xlsx"}

    for category, info in CONTRACT_CATEGORIES.items():
        expected_folder_name = info["path"].lower()
        matching_directories = [
            item
            for item in data_dir.rglob("*")
            if item.is_dir() and item.name.lower() == expected_folder_name
        ]

        span = trace.span(
            name=f"scan-{category}",
            input={
                "expected_folder": info["path"],
                "matching_directories": [str(directory) for directory in matching_directories]
            }
        )

        for category_directory in matching_directories:
            for item in category_directory.rglob("*"):
                if item.is_file() and item.suffix.lower() in supported_extensions:
                    discovered[category].append(item)

        span.end(
            output={
                "files_found": len(discovered[category]),
                "status": "success"
            }
        )

    total_files = sum(len(files) for files in discovered.values())
    categories_with_files = sum(1 for files in discovered.values() if files)
    trace.update(
        output={
            "status": "success",
            "total_files": total_files,
            "categories_with_files": categories_with_files,
            "files_per_category": {
                category: len(files)
                for category, files in discovered.items()
            }
        }
    )

    return discovered


# Search upward so this works whether Jupyter starts in Research/, the project,
# or the workspace root. This project currently stores the bundle as
# ContractIQ_data_reserve, so both valid directory names are supported.
current_directory = Path.cwd().resolve()
candidate_data_dirs = []

for directory in [current_directory, *current_directory.parents]:
    candidate_data_dirs.extend([
        directory / "ContractIQ_data",
        directory / "ContractIQ_data_reserve",
        directory / "PWC_CONTRACTIQ_PROJECT" / "ContractIQ_data",
        directory / "PWC_CONTRACTIQ_PROJECT" / "ContractIQ_data_reserve"
    ])

DATA_DIR = next(
    (
        candidate
        for candidate in candidate_data_dirs
        if candidate.exists() and candidate.is_dir()
    ),
    None
)

if DATA_DIR is None:
    searched_locations = "\n".join(
        f"  - {candidate}"
        for candidate in candidate_data_dirs
    )
    raise FileNotFoundError(
        "Could not find the contract dataset. Checked these locations:\n"
        f"{searched_locations}"
    )

print(f"Notebook working directory: {current_directory}")
print(f"Using project data directory: {DATA_DIR}")
contract_files = discover_contract_data(DATA_DIR)

print("\nDiscovery Summary:")
total = 0

for category, files in contract_files.items():
    print(f"  {category}: {len(files)} files")
    total += len(files)

print(f"\nTotal files discovered: {total}")
langfuse.flush()

Notebook working directory: E:\Sheriff_faang\full_end_to_end_project_implementation\IK_PWC_Agentic_AI_Project\PWC_CAPSTONE_PROJECT\IK_PWC_COURSE_CAPSTONE_PROJECTS\PWC_CONTRACTIQ_PROJECT\Research
Using project data directory: E:\Sheriff_faang\full_end_to_end_project_implementation\IK_PWC_Agentic_AI_Project\PWC_CAPSTONE_PROJECT\IK_PWC_COURSE_CAPSTONE_PROJECTS\PWC_CONTRACTIQ_PROJECT\ContractIQ_data_reserve

Discovery Summary:
  master_agreements: 6 files
  transaction_contracts: 4 files
  commercial_docs: 2 files
  financial_billing: 4 files
  operational_docs: 4 files
  compliance_docs: 6 files
  historical_negotiation: 6 files
  disputes_audits_governance: 4 files

Total files discovered: 36


## Checkpoint: Week 1 Verification

Run this cell to verify your Week 1 implementation.

In [14]:
# ============================================================================
# WEEK 1 CHECKPOINT - VALIDATION
# ============================================================================

print("WEEK 1 CHECKPOINT - Verification")
print("=" * 60)

checks = []

# Check 1: Langfuse initialized
try:
    checks.append(("Langfuse initialized", langfuse is not None))
except:
    checks.append(("Langfuse initialized", False))

# Check 2: Session ID created
try:
    checks.append(("Session ID created", SESSION_ID is not None and len(SESSION_ID) > 10))
except:
    checks.append(("Session ID created", False))

# Check 3: OpenAI client ready
try:
    checks.append(("OpenAI client ready", openai_client is not None))
except:
    checks.append(("OpenAI client ready", False))

# Check 4: traced_embedding works
try:
    test_emb = traced_embedding("test", "test-validation")
    checks.append(("traced_embedding() works", test_emb is not None and len(test_emb) == 1536))
except Exception as e:
    checks.append(("traced_embedding() works", False))
    print(f"  Error: {e}")

# Check 5: traced_completion works
try:
    test_comp = traced_completion("Say 'hello'", "test-validation")
    checks.append(("traced_completion() works", test_comp is not None and len(test_comp) > 0))
except Exception as e:
    checks.append(("traced_completion() works", False))
    print(f"  Error: {e}")

# Check 6: Contract taxonomy
checks.append(("Contract taxonomy defined", len(CONTRACT_CATEGORIES) == 8))

# Check 7: Data discovery
try:
    checks.append(("Data discovery function works", callable(discover_contract_data)))
except:
    checks.append(("Data discovery function works", False))

# Display results
all_passed = True
for check_name, check_result in checks:
    status = "PASS" if check_result else "FAIL"
    print(f"  [{status}] {check_name}")
    if not check_result:
        all_passed = False

print("\n" + "=" * 60)
if all_passed:
    print("ALL WEEK 1 CHECKPOINTS PASSED! Ready for Week 2.")
else:
    print("Some checkpoints FAILED. Review your implementation above.")
    print("\nHint: Check the error messages and compare with expected output.")

WEEK 1 CHECKPOINT - Verification
  [PASS] Langfuse initialized
  [PASS] Session ID created
  [PASS] OpenAI client ready
  [PASS] traced_embedding() works
  [PASS] traced_completion() works
  [PASS] Contract taxonomy defined
  [PASS] Data discovery function works

ALL WEEK 1 CHECKPOINTS PASSED! Ready for Week 2.
